<a href="https://colab.research.google.com/github/Shivamrana0309/BE_Project_XRAY/blob/main/03_stage3_augmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import os
import cv2
import albumentations as A

# 1. Puraane folders se connection setup
preprocessed_dir = "/Users/shivam/Documents/Dataset/COVID-19_Radiography_Dataset_Preprocessed"
balanced_dir = "/Users/shivam/Documents/Dataset/COVID-19_Radiography_Dataset_Preprocessed"
classes = ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]

os.makedirs(balanced_dir, exist_ok=True)

# 2. Target Count nikalna
counts = {}
for cls in classes:
    cls_dir = os.path.join(preprocessed_dir, cls)
    counts[cls] = len(os.listdir(cls_dir)) if os.path.exists(cls_dir) else 0

target_count = max(counts.values())
print(f"Target images per class for balancing: {target_count}\n")

# 3. Augmentation Pipeline
aug_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0, scale_limit=0.10, rotate_limit=10, p=0.8),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.8)
])

# 4. Process and Generate Images
for cls in classes:
    input_class_dir = os.path.join(preprocessed_dir, cls)
    output_class_dir = os.path.join(balanced_dir, cls)
    os.makedirs(output_class_dir, exist_ok=True)

    if os.path.exists(input_class_dir):
        images = [f for f in os.listdir(input_class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        current_count = len(images)

        # ERROR FIX: Agar folder khali hai toh skip karo
        if current_count == 0:
            print(f"⚠️ WARNING: '{cls}' me 0 images hain! Augmentation possible nahi hai. Puraane step check karo.")
            continue

        print(f"Processing '{cls}' (Current Count: {current_count})...")

        # Step A: Base images ko nayi jagah copy karo
        for img_name in images:
            img_path = os.path.join(input_class_dir, img_name)
            image = cv2.imread(img_path)
            if image is not None:
                cv2.imwrite(os.path.join(output_class_dir, img_name), image)

        # Step B: Agar images kam hain, toh augment karo
        if current_count < target_count:
            needed = target_count - current_count
            print(f" -> Generating {needed} new variations...")

            for i in range(needed):
                img_name = images[i % current_count]
                img_path = os.path.join(input_class_dir, img_name)
                image = cv2.imread(img_path)

                if image is not None:
                    augmented = aug_pipeline(image=image)['image']
                    new_name = f"aug_{i}_{img_name}"
                    cv2.imwrite(os.path.join(output_class_dir, new_name), augmented)
        else:
            print(f" -> '{cls}' is already perfectly balanced.")

print(f"\nStage 3 Complete! Final dataset path: {balanced_dir}")

Target images per class for balancing: 10192

Processing 'COVID' (Current Count: 3616)...
 -> Generating 6576 new variations...
Processing 'Lung_Opacity' (Current Count: 6012)...
 -> Generating 4180 new variations...
Processing 'Normal' (Current Count: 10192)...
 -> 'Normal' is already perfectly balanced.
Processing 'Viral Pneumonia' (Current Count: 1345)...
 -> Generating 8847 new variations...

Stage 3 Complete! Final dataset path: /Users/shivam/Documents/Dataset/COVID-19_Radiography_Dataset_Preprocessed_Balanced
